# Deep Neural Networks(DNNs):

Deep Neural Networks (DNNs) are a class of Artificial Neural Networks (ANNs) characterized by having **multiple hidden layers** between the input and output layers. This "depth" allows them to learn hierarchical representations of data, extracting increasingly complex and abstract features at each successive layer. This capability is what enables DNNs to excel in tasks that are challenging for shallower models, such as image recognition, natural language processing, and complex pattern recognition. 🧠

-----

## Architecture and Forward Propagation

A DNN extends the concept of a Shallow Neural Network by incorporating two or more hidden layers. The information flows from the input layer, through each hidden layer, and finally to the output layer.

Let:

  * $n_x$: Number of features (input neurons)
  * $L$: Total number of layers in the network (excluding input layer). So, $L-1$ hidden layers.
  * $n^{[l]}$: Number of neurons in layer $l$.
  * $m$: Number of training examples.

### Mathematical Formulas (General Layer $l$ to $l+1$):

For any layer $l$ (where $l$ can be a hidden layer or the input layer feeding into the first hidden layer):

  * **Linear Combination (Weighted Sum):**
    $$Z^{[l]} = W^{[l]} A^{[l-1]} + B^{[l]}$$
    Where:

      * $A^{[l-1]}$: Activated output from the previous layer $l-1$. For the first hidden layer ($l=1$), $A^{[0]} = X$ (the input data matrix, shape $(n\_x, m)$).
      * $W^{[l]}$: Weight matrix for layer $l$ (neurons in layer $l \\times$ neurons in layer $l-1$, shape $(n^{[l]}, n^{[l-1]})$).
      * $B^{[l]}$: Bias vector for layer $l$ (neurons in layer $l \\times$ 1, shape $(n^{[l]}, 1)$).
      * $Z^{[l]}$: Linear combination output for layer $l$ (shape $(n^{[l]}, m)$).

  * **Activation Function:**
    $$A^{[l]} = g^{[l]}(Z^{[l]})$$
    Where:

      * $g^{[l]}$: Activation function for layer $l$. Common choices for hidden layers are ReLU, Tanh. The output layer's activation depends on the task (Sigmoid for binary classification, Softmax for multi-class classification, Linear for regression).
      * $A^{[l]}$: Activated output from layer $l$ (shape $(n^{[l]}, m)$).

This forward propagation process is repeated for each layer until the final output layer ($\hat{Y}$) is computed.

-----

## Key Components (Shared with Shallow Networks, but scale up)

  * **Weights ($W$) and Biases ($B$):** These are the learnable parameters of the network. There's a set of $W$ and $B$ for each layer.
  * **Activation Functions:** Introduce non-linearity, allowing the network to model complex relationships. Common ones:
      * **ReLU (Rectified Linear Unit)**: $g(z) = \\max(0, z)$ (popular in hidden layers).
      * **Sigmoid**: $g(z) = 1 / (1 + e^{-z})$ (output layer for binary classification).
      * **Softmax**: $g(\\mathbf{z})*i = \\frac{e^{z\_i}}{\\sum*{j=1}^{K} e^{z\_j}}$ (output layer for multi-class classification, outputs probabilities summing to 1).
      * **Tanh (Hyperbolic Tangent)**: $g(z) = \\frac{e^z - e^{-z}}{e^z + e^{-z}}$ (alternative for hidden layers, output range -1 to 1).
      * **Linear**: $g(z) = z$ (output layer for regression).
  * **Cost/Loss Function:** Quantifies the error between predicted and actual outputs.
      * **Binary Cross-Entropy**: For binary classification.
      * **Categorical Cross-Entropy**: For multi-class classification.
      * **Mean Squared Error (MSE)**: For regression.
  * **Optimization Algorithm (Gradient Descent and its variants):** Iteratively updates weights and biases to minimize the cost function. Popular variants like **Adam**, **RMSprop**, and **Adagrad** dynamically adjust the learning rate for faster and more stable convergence.
  * **Backpropagation:** The algorithm that efficiently calculates the gradients of the cost function with respect to all parameters by propagating the error backward through the network, layer by layer.
  * **Vectorization:** Crucial for efficient computation by utilizing highly optimized array operations provided by libraries like NumPy and executed on specialized hardware (GPUs).

-----

### Deep Neural Network for Classification

### Example Data

We'll use a more complex, non-linearly separable dataset (`make_circles`) to highlight where a deeper network can outperform a shallow one by learning more intricate features.


### Code and Explanation (Keras/TensorFlow)

Instead of building from scratch, we'll use **Keras (a high-level API for TensorFlow)**, which is the standard way to build DNNs. It significantly simplifies the process compared to pure NumPy, handling backpropagation, optimization, and parameter management automatically.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

print("--- Deep Neural Network for Classification (Keras/TensorFlow) ---")

# --- 1. Data Generation and Preprocessing ---
# Generate a more complex non-linearly separable dataset
X_cls, y_cls = make_circles(n_samples=1000, noise=0.05, factor=0.5, random_state=42)
y_cls = y_cls.reshape(-1, 1) # Ensure y is a column vector

# Split data
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

# Scale features. Crucial for DNNs for faster convergence.
scaler_cls = StandardScaler()
X_train_cls_scaled = scaler_cls.fit_transform(X_train_cls)
X_test_cls_scaled = scaler_cls.transform(X_test_cls)

print(f"X_train_cls_scaled shape: {X_train_cls_scaled.shape}")
print(f"y_train_cls shape: {y_train_cls.shape}")

# Plot raw data to visualize the complex non-linear separation
plt.figure(figsize=(8, 6))
plt.scatter(X_cls[y_cls.flatten() == 0, 0], X_cls[y_cls.flatten() == 0, 1], color='blue', label='Class 0', alpha=0.7)
plt.scatter(X_cls[y_cls.flatten() == 1, 0], X_cls[y_cls.flatten() == 1, 1], color='red', label='Class 1', alpha=0.7)
plt.title('Simulated Concentric Circles Dataset (Binary Classification)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True)
plt.show()

# --- 2. Build the Deep Neural Network Model ---
# Using Keras Sequential API: layers are stacked linearly.
model_cls = Sequential([
    # Input layer automatically inferred by input_shape in the first Dense layer.
    # First Hidden Layer: 128 neurons, ReLU activation
    Dense(128, activation='relu', input_shape=(X_train_cls_scaled.shape[1],)),
    # Second Hidden Layer: 64 neurons, ReLU activation
    Dense(64, activation='relu'),
    # Third Hidden Layer: 32 neurons, ReLU activation
    Dense(32, activation='relu'),
    # Output Layer: 1 neuron for binary classification, Sigmoid activation for probability output
    Dense(1, activation='sigmoid')
])

model_cls.summary() # Print a summary of the model's architecture

# --- 3. Compile the Model ---
# Configure the learning process
model_cls.compile(
    optimizer=Adam(learning_rate=0.001), # Adam optimizer is a popular choice for its efficiency
    loss='binary_crossentropy',         # Appropriate loss function for binary classification
    metrics=['accuracy']                # Metric to monitor during training
)

# --- 4. Train the Model ---
# Fit the model to the training data
history_cls = model_cls.fit(
    X_train_cls_scaled, y_train_cls,
    epochs=100,             # Number of times to iterate over the entire dataset
    batch_size=32,          # Number of samples per gradient update
    validation_split=0.1,   # Use 10% of training data for validation during training
    verbose=0               # Suppress verbose output during training, set to 1 or 2 for progress bars
)

# --- 5. Evaluate the Model ---
loss_cls, accuracy_cls = model_cls.evaluate(X_test_cls_scaled, y_test_cls, verbose=0)
print(f"\nTest Loss: {loss_cls:.4f}")
print(f"Test Accuracy: {accuracy_cls:.4f}")

# Make predictions (probabilities)
y_pred_prob_cls = model_cls.predict(X_test_cls_scaled)
# Convert probabilities to binary class labels (0 or 1)
y_pred_cls = (y_pred_prob_cls > 0.5).astype(int)

print("\nClassification Report:\n", classification_report(y_test_cls, y_pred_cls))

# Confusion Matrix to see correct vs. incorrect classifications per class
cm_cls = confusion_matrix(y_test_cls, y_pred_cls)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_cls, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for DNN Classification')
plt.show()

# Plot training history (loss and accuracy over epochs)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history_cls.history['loss'], label='Train Loss')
plt.plot(history_cls.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_cls.history['accuracy'], label='Train Accuracy')
plt.plot(history_cls.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Plotting the decision boundary
def plot_dnn_decision_boundary(model, X, y, scaler, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))

    # Scale meshgrid points before feeding to the model
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    grid_points_scaled = scaler.transform(grid_points)

    Z_prob = model.predict(grid_points_scaled, verbose=0)
    Z = (Z_prob > 0.5).astype(int)
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, cmap=plt.cm.RdBu, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y.flatten(), cmap=plt.cm.RdBu, edgecolors='k', s=20)
    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.show()

plot_dnn_decision_boundary(model_cls, X_cls, y_cls, scaler_cls, 'DNN Decision Boundary on Circles Dataset')

-----

## Deep Neural Network for Regression

### Example Data

We'll create a more complex, multi-dimensional non-linear regression problem.



### Code and Explanation (Keras/TensorFlow)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

print("\n--- Deep Neural Network for Regression (Keras/TensorFlow) ---")

# --- 1. Data Generation and Preprocessing ---
np.random.seed(42)
num_samples_reg = 1000
num_features_reg = 3

# Generate multi-dimensional input features
X_reg = np.random.rand(num_samples_reg, num_features_reg) * 10 - 5 # Features between -5 and 5

# Generate a complex non-linear target variable
# y = x1*sin(x2) + x3^2 + noise
y_reg = (X_reg[:, 0] * np.sin(X_reg[:, 1]) + X_reg[:, 2]**2 + np.random.normal(0, 0.5, num_samples_reg)).reshape(-1, 1)

# Split data
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Scale features
scaler_X_reg = StandardScaler()
X_train_reg_scaled = scaler_X_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_X_reg.transform(X_test_reg)

# Scale target variable (often beneficial for regression NNs)
scaler_y_reg = StandardScaler()
y_train_reg_scaled = scaler_y_reg.fit_transform(y_train_reg)
y_test_reg_scaled = scaler_y_reg.transform(y_test_reg)

print(f"X_train_reg_scaled shape: {X_train_reg_scaled.shape}")
print(f"y_train_reg_scaled shape: {y_train_reg_scaled.shape}")

# --- 2. Build the Deep Neural Network Model for Regression ---
model_reg = Sequential([
    # First Hidden Layer
    Dense(128, activation='relu', input_shape=(X_train_reg_scaled.shape[1],)),
    # Second Hidden Layer
    Dense(64, activation='relu'),
    # Third Hidden Layer
    Dense(32, activation='relu'),
    # Output Layer: 1 neuron for regression, no activation (linear activation by default)
    Dense(1)
])

model_reg.summary()

# --- 3. Compile the Model ---
model_reg.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse' # Mean Squared Error is standard for regression
)

# --- 4. Train the Model ---
history_reg = model_reg.fit(
    X_train_reg_scaled, y_train_reg_scaled,
    epochs=200,             # More epochs often needed for regression
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

# --- 5. Evaluate the Model ---
loss_reg = model_reg.evaluate(X_test_reg_scaled, y_test_reg_scaled, verbose=0)
print(f"\nTest Loss (Scaled MSE): {loss_reg:.4f}")

# Make predictions (scaled)
y_pred_reg_scaled = model_reg.predict(X_test_reg_scaled)

# Inverse transform predictions to original scale for meaningful evaluation
y_pred_reg = scaler_y_reg.inverse_transform(y_pred_reg_scaled)

# Evaluate on original scale
mse = mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"Mean Squared Error on Test Set (Original Scale): {mse:.4f}")
print(f"R-squared on Test Set (Original Scale): {r2:.4f}")

# Plot training history (loss over epochs)
plt.figure(figsize=(8, 5))
plt.plot(history_reg.history['loss'], label='Train Loss')
plt.plot(history_reg.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss Over Epochs (Regression)')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()

# Visualize predictions vs actual (for one feature for simplicity)
plt.figure(figsize=(10, 6))
# For visualization, pick one feature (e.g., X[:, 0]) and plot actual vs predicted.
# This is an approximate visualization as y depends on all 3 features.
plt.scatter(X_test_reg[:, 0], y_test_reg, label='Actual Y', alpha=0.7)
plt.scatter(X_test_reg[:, 0], y_pred_reg, label='Predicted Y', alpha=0.7, color='red')
plt.title('Actual vs Predicted Values (Slice by Feature 1)')
plt.xlabel('Feature 1 (Original Scale)')
plt.ylabel('Target Y (Original Scale)')
plt.legend()
plt.grid(True)
plt.show()

# Further visualization (e.g., residual plot)
residuals = y_test_reg - y_pred_reg
plt.figure(figsize=(10, 6))
plt.scatter(y_pred_reg, residuals, alpha=0.7)
plt.axhline(y=0, color='r', linestyle='--')
plt.title('Residual Plot (Prediction vs Residuals)')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals (Actual - Predicted)')
plt.grid(True)
plt.show()

-----

## Explanation of Code and Concepts

The shift from "from scratch" NumPy implementations to **Keras/TensorFlow** is significant because it's how deep learning is primarily done in practice. These libraries abstract away the complex manual implementation of backpropagation, gradient calculations, and parameter updates, allowing you to focus on model architecture and data.

### Key Aspects in Keras/TensorFlow Code:

1.  **Data Preprocessing**:

      * `StandardScaler`: Remains critical. Features are scaled to have zero mean and unit variance. For regression, scaling the target variable (`y`) can also help stabilize training, but remember to `inverse_transform` predictions for evaluation on the original scale.

2.  **Building the Model (`tf.keras.models.Sequential`)**:

      * `Sequential` is the simplest way to build a Keras model, stacking layers one after another.
      * `Dense` Layer: Represents a **fully connected layer**.
          * `units`: The number of neurons in that layer. This is `n^[l]` from our formulas.
          * `activation`: The activation function applied after the linear transformation (`g^[l]`).
              * **`'relu'`**: Standard for hidden layers.
              * **`'sigmoid'`**: For binary classification output (1 neuron).
              * **(None)** or **linear activation**: For regression output (1 neuron). Keras `Dense` layers default to linear activation if none is specified.
          * `input_shape`: Only needed for the *first* layer to inform the model about the dimensionality of your input data. Keras then automatically infers the input shape for subsequent layers.
      * **Depth**: The presence of multiple `Dense` layers *before* the output layer defines the network as "deep." Each `Dense` layer here represents a hidden layer.

3.  **Compiling the Model (`model.compile`)**:

      * `optimizer`: Specifies the algorithm used to update the model's weights and biases during training. **Adam** is highly recommended and widely used due to its adaptive learning rate capabilities, making it robust and efficient. Other options include SGD, RMSprop, Adagrad, etc.
          * `learning_rate`: The step size for the optimizer.
      * `loss`: The function the model tries to minimize.
          * **`'binary_crossentropy'`**: For binary classification when the output layer has a sigmoid activation.
          * **`'mse'`**: For regression.
          * (`'categorical_crossentropy'` for multi-class classification with one-hot encoded labels, or `'sparse_categorical_crossentropy'` for integer labels).
      * `metrics`: A list of metrics to be evaluated by the model during training and testing. These are for monitoring and don't directly influence the optimization process. `'accuracy'` is common for classification.

4.  **Training the Model (`model.fit`)**:

      * `epochs`: The number of times the training algorithm will iterate over the entire training dataset. More epochs can lead to better learning but also **overfitting**.
      * `batch_size`: The number of samples processed before the model's parameters are updated. This is **mini-batch gradient descent**.
      * `validation_split`: Keras can automatically set aside a portion of the training data as a validation set. Monitoring `val_loss` and `val_accuracy` helps detect overfitting (when training loss continues to drop but validation loss starts to rise).
      * `verbose`: Controls the verbosity of the training output.

5.  **Evaluation (`model.evaluate`, `model.predict`)**:

      * `model.evaluate()`: Returns the loss and any specified metrics (e.g., accuracy) on a given dataset (typically the test set).
      * `model.predict()`: Generates predictions. For classification, it outputs probabilities which you then threshold to get class labels. For regression, it outputs the predicted continuous values.
      * **Metrics**: Standard metrics like `accuracy_score`, `classification_report`, `confusion_matrix` for classification, and `mean_squared_error`, `r2_score` for regression are used from `sklearn.metrics`.

### Why Deep?

The multi-layered structure of DNNs allows them to learn:

  * **Hierarchical Features**: Earlier layers learn simple features (edges, textures in images; sounds in audio), while later layers combine these into more complex, abstract representations (parts of objects, words, phrases).
  * **Increased Capacity**: More layers and neurons allow the network to model highly complex and non-linear relationships that shallower models cannot.
  * **Automatic Feature Engineering**: DNNs reduce the need for manual feature engineering, as they can learn relevant features directly from raw data.

Understanding these concepts and seeing them implemented with a framework like Keras provides a strong foundation for building and training your own deep neural networks for various tasks.